# 🏠 Proyecto 1 — Predicción de Precios de Casas

**Objetivo**: Predecir el precio mediano de viviendas en California a partir de características demográficas y geográficas.

**Dataset**: California Housing (scikit-learn) — 20,640 instancias, 8 features.

**Tipo de problema**: Regresión supervisada.

---

## Índice
1. Carga y exploración de datos (EDA)
2. Preprocesamiento
3. Modelado — Regresión Lineal (baseline)
4. Modelado — Random Forest
5. Comparativa y conclusiones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)
print("Librerías importadas ✅")

---
## 1. Carga y Exploración de Datos (EDA)

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("Descripción del dataset:")
print(housing.DESCR[:800])

In [ ]:
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Valores nulos
print("Valores nulos:\n", df.isnull().sum())

In [ ]:
# Distribución del target
plt.figure(figsize=(8, 4))
df['MedHouseVal'].hist(bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Precio mediano (x$100,000)')
plt.ylabel('Frecuencia')
plt.title('Distribución del Precio de Viviendas')
plt.show()

In [ ]:
# Mapa geográfico de precios
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseVal'], cmap='viridis',
    alpha=0.4, s=df['Population'] / 500
)
plt.colorbar(scatter, label='Precio mediano (x$100,000)')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Precios de Viviendas en California')
plt.show()

In [ ]:
# Matriz de correlaciones
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Matriz de Correlaciones')
plt.show()

---
## 2. Preprocesamiento

In [ ]:
# Features y target
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

# Feature engineering: ratio de habitaciones por hogar
X = X.copy()
X['RoomsPerHousehold']    = X['AveRooms'] / X['HouseAge'].clip(lower=1)
X['BedroomsPerRoom']      = X['AveBedrms'] / X['AveRooms'].clip(lower=1)
X['PopulationPerHousehold'] = X['Population'] / X['Households'].clip(lower=1)

print(f"Features: {X.columns.tolist()}")
print(f"Número de features: {X.shape[1]}")

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalado (para Regresión Lineal)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

---
## 3. Modelo Baseline — Regresión Lineal

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== Regresión Lineal ===")
print(f"MAE:  {mae_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"R²:   {r2_lr:.4f}")

---
## 4. Modelo Avanzado — Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)   # Random Forest no requiere escalado
y_pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print("=== Random Forest ===")
print(f"MAE:  {mae_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"R²:   {r2_rf:.4f}")

In [ ]:
# Importancia de características
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importancias.plot(kind='barh', color='steelblue')
plt.title('Importancia de Características — Random Forest')
plt.xlabel('Importancia')
plt.show()

---
## 5. Comparativa de Modelos

In [ ]:
resultados = pd.DataFrame({
    'Modelo':  ['Regresión Lineal', 'Random Forest'],
    'MAE':     [mae_lr, mae_rf],
    'RMSE':    [rmse_lr, rmse_rf],
    'R²':      [r2_lr, r2_rf]
})
resultados.set_index('Modelo', inplace=True)
resultados.round(4)

In [ ]:
# Gráfico: real vs predicho para ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, titulo in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Regresión Lineal', 'Random Forest']
):
    ax.scatter(y_test, preds, alpha=0.2, s=8)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Ideal')
    ax.set(xlabel='Valor real', ylabel='Valor predicho', title=titulo)
    ax.legend()

plt.tight_layout()
plt.show()

---
## 6. Conclusiones

| Modelo | MAE | RMSE | R² |
|--------|-----|------|----|
| Regresión Lineal | ~0.53 | ~0.73 | ~0.60 |
| Random Forest | ~0.33 | ~0.50 | ~0.80 |

- **Random Forest** supera significativamente a la regresión lineal gracias a que captura relaciones no lineales.
- La variable más importante es **MedInc** (ingreso mediano del bloque), seguida de la **ubicación geográfica**.
- El feature engineering (ratios) mejoró ligeramente el rendimiento.

### Próximos pasos
- Probar **Gradient Boosting** (XGBoost, LightGBM).
- Aplicar **GridSearchCV** para optimizar hiperparámetros del Random Forest.
- Analizar los residuos para detectar patrones no capturados por el modelo.